In [2]:
#@title 1. Connect to Google Drive & Define Data Paths
from google.colab import drive
import os

# Prompt to authorize Colab to access your Google Drive
drive.mount('/content/drive')
#os.chdir('/content/drive/MyDrive/NPL2025_Proj')

# Define the path to data folder in Google Drive
save_dir = '/content/drive/MyDrive/NPL2025_Proj'

# Define the full paths for your input and output files
TRAIN_FILE = os.path.join(save_dir, "train_rehydrated.jsonl")
TEST_FILE = os.path.join(save_dir, "dev_rehydrated.jsonl")

# Verify files exist
if os.path.exists(TRAIN_FILE) and os.path.exists(TEST_FILE):
    print("Successfully located rehydrated files in Google Drive.")
    print(f"Training file: {TRAIN_FILE}")
    print(f"Test file:     {TEST_FILE}")
else:
    print("Error: Could not find 'train_rehydrated.jsonl' or 'dev_rehydrated.jsonl' at the specified path.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully located rehydrated files in Google Drive.
Training file: /content/drive/MyDrive/NPL2025_Proj/train_rehydrated.jsonl
Test file:     /content/drive/MyDrive/NPL2025_Proj/dev_rehydrated.jsonl


In [3]:
# 1. Uninstall the conflicting library
#!pip uninstall -y peft

# 2. Re-run your install (including pandas from the previous fix)
!pip install -q numpy==1.26.4 transformers==4.41.2 datasets==2.19.1 accelerate==0.31.0 torch==2.3.1 scikit-learn==1.6.0 pandas

In [1]:
#@title 2. Setup Environment and Install Dependencies
# Install the required Python libraries
#!pip install -q numpy==1.26.4 transformers==4.41.2 datasets==2.19.1 accelerate==0.31.0 torch==2.3.1 scikit-learn==1.6.0

import json
import sys
import os
import glob
import zipfile
from collections import defaultdict
from typing import List, Dict, Any

import numpy as np
from datasets import Dataset, disable_progress_bar
from transformers import (
    RobertaTokenizerFast,
    RobertaForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification,
    TrainingArguments,
)
from google.colab import files

# Disable the progress bars
disable_progress_bar()

print("Environment setup complete.")

Environment setup complete.


In [3]:
#@title 3. Configuration and Helper Functions
# This cell contains all the configuration variables and helper functions.

# Main Configuration
MODEL_NAME = "roberta-base"
OUTPUT_DIR_BASE = "roberta-single-type-simplified"
MARKER_TYPES = ["Action", "Actor", "Effect", "Evidence", "Victim"]
TEMP_SUBMISSION_FILE = "submission.jsonl"

# The submission.zip file will be saved in Drive folder
FINAL_SUBMISSION_ZIP = os.path.join(save_dir, "submission.zip")


# Training Hyperparameters
TRAIN_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10

# Inference Hyperparameters
INFER_BATCH_SIZE = 64

# --- All Helper Functions ---
def load_data(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                item["_id"] = item.get("_id", f"sample_{len(data)}")
                item["text"] = item.get("text", "")
                item["markers"] = item.get("markers", [])
                item["conspiracy"] = item.get("conspiracy", "No")
                data.append(item)
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line: {line.strip()}")
    return data

def create_label_maps_simplified(marker_type):
    label_list = ["O", marker_type]
    label_to_id = {label: i for i, label in enumerate(label_list)}
    id_to_label = {i: label for label, i in label_to_id.items()}
    return label_to_id, id_to_label, len(label_list)

def tokenize_and_align_labels_for_training(examples, tokenizer, label_to_id, marker_type):
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128, return_offsets_mapping=True)
    labels = []
    all_markers = examples.get("markers", [])
    for i, offsets in enumerate(tokenized_inputs["offset_mapping"]):
        example_labels = [0] * len(offsets)
        example_markers = all_markers[i] if i < len(all_markers) else []
        for marker in example_markers:
            if marker["type"] == marker_type:
                start_char, end_char = marker["startIndex"], marker["endIndex"]
                marker_label = label_to_id.get(marker_type)
                if marker_label is not None:
                    for token_idx, (start, end) in enumerate(offsets):
                        if start is not None and end is not None:
                            if start_char <= start < end_char or (start < end_char and end > start_char):
                                if token_idx < len(example_labels):
                                    example_labels[token_idx] = marker_label
        labels.append(example_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def tokenize_for_inference(examples, tokenizer):
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128, return_offsets_mapping=True)
    tokenized_inputs["labels"] = [[-100] * len(offset_map) for offset_map in tokenized_inputs["offset_mapping"]]
    return tokenized_inputs

def find_latest_checkpoint(base_path, marker_type):
    full_path = f"{base_path}-{marker_type}"
    checkpoint_dirs = glob.glob(os.path.join(full_path, "checkpoint-*"))
    if not checkpoint_dirs: return full_path
    checkpoint_dirs.sort(key=lambda x: int(os.path.basename(x).split('-')[-1]))
    return checkpoint_dirs[-1]

def reconstruct_spans(predictions, tokenized_dataset, id_to_label):
    reconstructed_markers = defaultdict(list)
    positive_label_type = id_to_label.get(1)
    if not positive_label_type or positive_label_type == "O": return reconstructed_markers
    for i, pred_ids in enumerate(predictions):
        offsets = tokenized_dataset[i]['offset_mapping']
        original_text = tokenized_dataset[i]['text']
        current_span_start_char = None
        for token_idx, label_id in enumerate(pred_ids):
            offset_tuple = offsets[token_idx]
            is_special = not offset_tuple or offset_tuple[0] == offset_tuple[1]
            if current_span_start_char is not None and (is_special or id_to_label[label_id] == 'O'):
                prev_end_char = offsets[token_idx - 1][1]
                span_text = original_text[current_span_start_char:prev_end_char]
                reconstructed_markers[i].append({"startIndex": current_span_start_char, "endIndex": prev_end_char, "type": positive_label_type, "text": span_text})
                current_span_start_char = None
            if current_span_start_char is None and not is_special and id_to_label[label_id] == positive_label_type:
                current_span_start_char = offset_tuple[0]
        if current_span_start_char is not None:
            last_valid_end = [o[1] for o in offsets if o and o[1] is not None][-1]
            span_text = original_text[current_span_start_char:last_valid_end]
            reconstructed_markers[i].append({"startIndex": current_span_start_char, "endIndex": last_valid_end, "type": positive_label_type, "text": span_text})
    return reconstructed_markers

def save_and_zip(file_path: str, data: List[Dict], output_zip_path: str):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data: f.write(json.dumps(item) + '\n')
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(file_path, arcname=os.path.basename(file_path))
    os.remove(file_path)
    print(f"Successfully created final submission file and saved to Google Drive: {output_zip_path}")

print("Configuration and helper functions are defined for RoBERTa.")



Configuration and helper functions are defined for RoBERTa.


In [4]:
#@title 4. Train the RoBERTa Models
# This cell will train five separate RoBERTa models, one for each marker type.

# Import RoBERTa-specific classes
from transformers import RobertaTokenizerFast, RobertaForTokenClassification

# Load data once
train_data = load_data(TRAIN_FILE)
train_dataset = Dataset.from_list(train_data)
# RoBERTa's tokenizer often works best with add_prefix_space=True
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)

for marker_type in MARKER_TYPES:
    print(f"\n--- Training model for marker type: {marker_type} ---")

    # Create label maps for the current marker type
    label_to_id, id_to_label, num_labels = create_label_maps_simplified(marker_type)

    # Tokenize and align labels
    tokenized_train_dataset = train_dataset.map(
        tokenize_and_align_labels_for_training,
        batched=True,
        fn_kwargs={"tokenizer": tokenizer, "label_to_id": label_to_id, "marker_type": marker_type}
    )

    # Load a new RoBERTa model for each marker type
    model = RobertaForTokenClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

    # Define training arguments
    output_dir = f"{OUTPUT_DIR_BASE}-{marker_type}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=len(tokenized_train_dataset) // TRAIN_BATCH_SIZE,
        report_to="none",
        save_strategy="epoch",
        load_best_model_at_end=False,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    # Train the model
    trainer.train()
    print(f"Training for {marker_type} finished.")

print("\n All RoBERTa models have been trained successfully!")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

Step,Training Loss
269,0.258300
538,0.225500
807,0.193000
1076,0.155000
1345,0.127300
1614,0.105200
1883,0.092800
2152,0.081400
2421,0.071200
2690,0.066000


Training for Evidence finished.

--- Training model for marker type: Victim ---


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
269,0.110400
538,0.073900
807,0.053600
1076,0.041600
1345,0.034100
1614,0.028000
1883,0.023800
2152,0.020600
2421,0.018400
2690,0.016700


Training for Victim finished.

 All RoBERTa models have been trained successfully!


In [5]:
#@title 5. Run Inference and Generate Submission File
# This cell uses the five trained RoBERTa models to predict markers on the dev set,
# aggregates the results, and creates the final 'submission.zip' file.

# Load test data
raw_data = load_data(TEST_FILE)
if not raw_data:
    print("Error: No test data loaded. Cannot perform inference.")
else:
    unique_ids = [d["_id"] for d in raw_data]
    conspiracy_keys = [d["conspiracy"] for d in raw_data]
    test_dataset = Dataset.from_list(raw_data)

    # Tokenize test data
    tokenized_test_dataset = test_dataset.map(
        tokenize_for_inference,
        batched=True,
        remove_columns=[col for col in test_dataset.column_names if col not in ['text', 'offset_mapping', '_id', 'conspiracy']],
        fn_kwargs={"tokenizer": tokenizer}
    )

    all_predicted_markers = defaultdict(list)

    # Iterate and infer for each marker type
    for marker_type in MARKER_TYPES:
        print(f"\n--- Running inference for type: {marker_type} ---")
        model_directory = find_latest_checkpoint(OUTPUT_DIR_BASE, marker_type)
        print(f"Loading model from: {model_directory}")

        try:
            # Load the RoBERTa model
            model = RobertaForTokenClassification.from_pretrained(model_directory)
            id_to_label = {0: "O", 1: marker_type}
        except Exception as e:
            print(f"Error loading model for {marker_type}. Details: {e}")
            continue

        # Prepare for inference
        prediction_args = Trainer(
            model=model,
            args=TrainingArguments(output_dir=f"./tmp_inference", per_device_eval_batch_size=INFER_BATCH_SIZE, report_to="none"),
            data_collator=DataCollatorForTokenClassification(tokenizer),
            tokenizer=tokenizer
        )

        # Perform inference
        predictions_output = prediction_args.predict(tokenized_test_dataset)
        predicted_class_ids = np.argmax(predictions_output.predictions, axis=2)

        # Reconstruct and aggregate spans
        current_marker_map = reconstruct_spans(predicted_class_ids, tokenized_test_dataset, id_to_label)
        for i, markers in current_marker_map.items():
            all_predicted_markers[i].extend(markers)

    # Assemble final submission objects
    jsonl_lines = []
    for i in range(len(raw_data)):
        jsonl_obj = {
            "_id": unique_ids[i],
            "conspiracy": conspiracy_keys[i],
            "markers": all_predicted_markers.get(i, [])
        }
        jsonl_lines.append(jsonl_obj)

    # Save and zip the result
    save_and_zip(TEMP_SUBMISSION_FILE, jsonl_lines, FINAL_SUBMISSION_ZIP)

    print("\n🎉 Inference complete!")



--- Running inference for type: Action ---
Loading model from: roberta-single-type-simplified-Action/checkpoint-2700



--- Running inference for type: Actor ---
Loading model from: roberta-single-type-simplified-Actor/checkpoint-2700



--- Running inference for type: Effect ---
Loading model from: roberta-single-type-simplified-Effect/checkpoint-2700



--- Running inference for type: Evidence ---
Loading model from: roberta-single-type-simplified-Evidence/checkpoint-2700



--- Running inference for type: Victim ---
Loading model from: roberta-single-type-simplified-Victim/checkpoint-2700


Successfully created final submission file and saved to Google Drive: /content/drive/MyDrive/NPL2025_Proj/submission.zip

🎉 Inference complete!
